# Kisan Mitra — run all experiments (Colab, T4 GPU)

This notebook runs the **complete evaluation** of the Kisan Mitra farmer advisory assistant and packs every result into
`kisan_mitra_results.zip`.

**Before you start**
1. *Runtime → Change runtime type → **T4 GPU*** (free tier is enough).
2. Upload the whole `kisan-mitra` folder to **My Drive** (so the path is `MyDrive/kisan-mitra`).
3. Add your Hugging Face token as a Colab secret named `HF_TOKEN`, and accept the terms of the 3 gated AI4Bharat models (links in cell 4).
4. *Runtime → Run all*. Expected time: **~60–90 min** (most of it is downloading ~20 GB of model weights and generating ~200 answers).

**If Colab disconnects:** reconnect, run cells 2, 4 and then cell 6 again — finished work is saved in `results/` on Drive and is skipped.

When it finishes, the zip downloads automatically (and is also saved in your Drive folder). Bring it back to your laptop.

In [5]:
# 1) Check the GPU
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

name, memory.total [MiB], memory.used [MiB]
Tesla T4, 15360 MiB, 3 MiB


In [6]:
# 2) Mount Google Drive and point to the uploaded project folder
from google.colab import drive
drive.mount('/content/drive')

import os, sys
PROJECT_DIR = '/content/drive/MyDrive/kisan-mitra'   # <-- change this if you uploaded the folder elsewhere
assert os.path.exists(f'{PROJECT_DIR}/configs/default.yaml'), \
    f'Could not find the project at {PROJECT_DIR}. Upload the whole kisan-mitra folder to "My Drive" first.'
os.chdir(PROJECT_DIR)
if f'{PROJECT_DIR}/src' not in sys.path:
    sys.path.insert(0, f'{PROJECT_DIR}/src')
print('Project folder:', PROJECT_DIR)
print(sorted(os.listdir(PROJECT_DIR)))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Project folder: /content/drive/MyDrive/kisan-mitra
['.env', '.gitignore', 'LICENSE', 'README.md', 'configs', 'data', 'docs', 'notebooks', 'pyproject.toml', 'reports', 'requirements-asr.txt', 'requirements-dev.txt', 'requirements.txt', 'results', 'scripts', 'src', 'tests']


In [7]:
# 3) Install dependencies (~2-3 min). Colab already has torch / torchaudio.
!pip install -q -r requirements.txt
!pip install -q -r requirements-asr.txt || pip install -q onnxruntime-gpu onnx
!apt-get -qq install -y ffmpeg > /dev/null
print('\nDone. If pip printed "you must restart the runtime": Runtime > Restart session, then run cell 2 again and continue from cell 4.')

ERROR: Could not find a version that satisfies the requirement onnxruntime-gpu==1.20.1 (from versions: 1.20.0, 1.20.2, 1.21.0, 1.21.1, 1.22.0, 1.23.0, 1.23.2, 1.24.1, 1.24.4, 1.25.0, 1.25.1, 1.26.0, 1.27.0, 1.28.0, 1.29.0, 1.30.0, 1.31.0, 1.31.1)
ERROR: No matching distribution found for onnxruntime-gpu==1.20.1

Done. If pip printed "you must restart the runtime": Runtime > Restart session, then run cell 2 again and continue from cell 4.


In [8]:
# 4) Hugging Face token + model access check
#    - Add your token: left sidebar "key" icon (Secrets) > Add new secret > Name: HF_TOKEN, Value: hf_xxx > enable "Notebook access"
#    - Accept the terms (one click each, while logged in) on:
#        https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M
#        https://huggingface.co/ai4bharat/indictrans2-en-indic-dist-200M
#        https://huggingface.co/ai4bharat/indic-conformer-600m-multilingual
from kisan_mitra.config import load_config
from kisan_mitra.hf_utils import hf_token, check_model_access, env_report

cfg = load_config()
print('HF token found' if hf_token() else 'NO HF_TOKEN FOUND - add it in Colab Secrets (see above)')
status = check_model_access(list(cfg['models'].values()))
for m, s in status.items():
    print(('OK   ' if s == 'ok' else 'FAIL ') + m + ('' if s == 'ok' else '  ->  ' + s))
print(env_report())
bad = [m for m, s in status.items() if s != 'ok']
assert not bad, 'Fix model access for: ' + ', '.join(bad)

HF token found


config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

OK   bharatgenai/AgriParam
OK   bharatgenai/Param-1-2.9B-Instruct
OK   ai4bharat/indictrans2-indic-en-dist-200M
OK   ai4bharat/indictrans2-en-indic-dist-200M
OK   ai4bharat/indic-conformer-600m-multilingual
OK   intfloat/multilingual-e5-base
OK   cross-encoder/ms-marco-MiniLM-L-6-v2
OK   MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli
{'torch': '2.11.0+cu130', 'cuda': '13.0', 'gpu': 'Tesla T4', 'gpu_mem_gb': 15.6, 'transformers': '4.56.2', 'sentence_transformers': '5.7.0', 'IndicTransToolkit': 'installed', 'gradio': '6.17.3', 'onnxruntime': '1.31.1'}


### 5) Smoke test (≈2 min)
Loads IndicTrans2 and translates one sentence each way, so problems show up before the long run.

In [9]:
from kisan_mitra.factory import build_translator
from kisan_mitra.hf_utils import free_memory
tr = build_translator(cfg)
print(tr('धान की नर्सरी में बीज उपचार के लिए कार्बेन्डाजिम की कितनी मात्रा प्रयोग करें?', 'hi', 'en'))
print(tr('ರಾಗಿ ಸಸಿಗಳನ್ನು ಎಷ್ಟು ದಿನಗಳ ವಯಸ್ಸಿನಲ್ಲಿ ನಾಟಿ ಮಾಡಬೇಕು?', 'kn', 'en'))
print(tr.translate_long('Use 20 kg of seed per hectare. Transplant 25-day-old seedlings.', 'en', 'ta'))
del tr; free_memory()

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_indictrans.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-indic-en-dist-200M:
- tokenization_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


dict.SRC.json: 0.00B [00:00, ?B/s]

dict.TGT.json: 0.00B [00:00, ?B/s]

model.SRC:   0%|          | 0.00/3.26M [00:00<?, ?B/s]

... (18 outputs left out)

A new version of the following files was downloaded from https://huggingface.co/ai4bharat/indictrans2-en-indic-dist-200M:
- modeling_indictrans.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

How much of carbendazim should be used for seed treatment in a paddy nursery?
At what age should the millet plants be transplanted?
ஒரு ஹெக்டேருக்கு 20 கிலோ விதைகளைப் பயன்படுத்துங்கள். 25 நாட்கள் பழமையான நாற்றுகளை மாற்று நடவு செய்யுங்கள்.


### 6) Run everything
Stages: question translation → retrieval ablation → speech recognition on spoken questions → 5 systems + voice run
(extractive RAG, AgriParam closed-book, AgriParam RAG, **Kisan Mitra full agent**, Param-1 RAG) → scoring.
Progress is printed per question. Safe to re-run: completed work is skipped.

In [10]:
from kisan_mitra.experiment import run_all
out = run_all(results_dir=f'{PROJECT_DIR}/results')

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

... (448 outputs left out)

model-00001-of-00002.safetensors:   0%|          | 0.00/4.67G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

[bharatgenai/Param-1-2.9B-Instruct] generate(use_cache=True) failed (AttributeError: 'NoneType' object has no attribute 'shape'); retrying with use_cache=False
  [param1_rag] 1/40 en01 route=rag t=3.02s :: 'For hybrid rice, you should use 20 kg per hectare.'
  [param1_rag] 2/40 en02 route=rag t=13.09s :: 'In SRI rice, transplant one or two 15-day-old seedlings per hill within 30 minutes of upro'
  [param1_rag] 3/40 en03 route=rag t=4.03s :: 'The economic threshold level for pink bollworm using pheromone traps is 8 moths per trap p'
  [param1_rag] 4/40 en04 route=rag t=9.46s :: 'Vermicompost takes 45 to 50 days to be ready, and Eisenia foetida (red earthworm) is prefe'
  [param1_rag] 5/40 en05 route=rag t=4.68s :: 'A farmer pays a maximum premium of 2% of the sum insured for Kharif crops under PMFBY.'
  [param1_rag] 6/40 en06 route=rag t=6.0s :: 'The Kisan Call Centre is open from 6:00 am to 10:00 pm on all seven days of the week.'
  [param1_rag] 7/40 en07 route=rag t=9.01s :: 'Late bli

### 7) Quick look at the results

In [11]:
import pandas as pd
pd.set_option('display.max_columns', 50); pd.set_option('display.width', 250)
R = f'{PROJECT_DIR}/results'
show = ['system','task_success','qa_all_facts_acc','qa_fact_recall','qa_groundedness','tool_quantity_acc',
        'ood_abstain_recall','qa_false_abstain_rate','route_acc','success_en','success_hi','success_kn','success_ta',
        'latency_p50_s','errors']
display(pd.read_csv(f'{R}/metrics_by_system.csv')[show])
display(pd.read_csv(f'{R}/retrieval_ablation.csv'))
display(pd.read_csv(f'{R}/translation_summary.csv'))
import glob
for f in glob.glob(f'{R}/asr_summary_*.csv'):
    display(pd.read_csv(f))

,system,task_success,qa_all_facts_acc,qa_fact_recall,qa_groundedness,tool_quantity_acc,ood_abstain_recall,qa_false_abstain_rate,route_acc,success_en,success_hi,success_kn,success_ta,latency_p50_s,errors
0,agriparam_closedbook,0.2000,0.2667,0.3778,0.3667,0.0000,0.0,0.0000,NaN,0.2500,0.2500,0.000,0.250,2.62,0
1,agriparam_rag,0.7000,0.9000,0.9167,0.6333,0.2000,0.0,0.0000,NaN,0.6667,0.8333,0.625,0.625,7.14,0
2,extractive_rag,0.6250,0.8333,0.8667,1.0000,0.0000,0.0,0.0000,NaN,0.4167,0.8333,0.625,0.625,0.90,0
3,kisan_mitra_full,0.8750,0.9000,0.9167,0.6609,0.6000,1.0,0.0333,0.9250,0.9167,1.0000,0.750,0.750,5.62,0
4,kisan_mitra_full_voice,0.7857,0.8182,0.8561,0.6746,0.3333,1.0,0.0455,0.8929,NaN,0.8333,0.750,0.750,5.68,0
5,param1_rag,0.5500,0.7333,0.7833,0.8278,0.0000,0.0,0.0000,NaN,0.5833,0.5833,0.500,0.500,7.19,0


,mode,query,n,hit@1,hit@3,hit@5,mrr
0,bm25,mt_en,35,0.9143,0.9714,0.9714,0.9381
1,bm25,reference_en,35,0.9429,1.0000,1.0000,0.9667
2,dense,mt_en,35,0.9429,1.0000,1.0000,0.9667
3,dense,reference_en,35,0.9143,1.0000,1.0000,0.9524
4,dense,original_dense,35,0.8571,0.9714,0.9714,0.9095
5,hybrid,mt_en,35,0.9429,0.9714,1.0000,0.9643
6,hybrid,reference_en,35,0.9714,1.0000,1.0000,0.9857
7,hybrid,original_dense,35,0.8857,1.0000,1.0000,0.9381
8,hybrid_rerank,mt_en,35,0.9714,1.0000,1.0000,0.9857
9,hybrid_rerank,reference_en,35,0.9429,1.0000,1.0000,0.9667


,lang,n,corpus_chrf++
0,hi,12,61.01
1,kn,8,61.80
2,ta,8,57.07
3,all,28,60.07


,source,decoding,lang,n,wer,cer,errors
0,gtts,ctc,hi,12,0.0723,0.0516,0
1,gtts,ctc,kn,8,0.2860,0.0557,0
2,gtts,ctc,ta,8,0.1141,0.0378,0
3,gtts,ctc,all,28,0.1453,0.0488,0
4,gtts,rnnt,hi,12,0.0986,0.0572,0
5,gtts,rnnt,kn,8,0.3016,0.0588,0
6,gtts,rnnt,ta,8,0.0962,0.0350,0
7,gtts,rnnt,all,28,0.1559,0.0513,0


### 8) Zip and download the results

In [12]:
from kisan_mitra.evaluate import zip_results
z = zip_results(f'{PROJECT_DIR}/results', f'{PROJECT_DIR}/kisan_mitra_results')
print('Saved to Drive:', z)
from google.colab import files
files.download(z)

Saved to Drive: /content/drive/MyDrive/kisan-mitra/kisan_mitra_results.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>